# S9 - Sistemas expertos (Pagarium) — Soluciones

Cuaderno **de soluciones** de la S9 (RA5-a/c). Resuelto y con la salida real en cada celda. Se corresponde con `sesion09_sistemas_expertos.ipynb`.

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre |
| `{ "clave": "valor" }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: valores ordenados |
| `def nombre(...):` / `class Nombre:` | Crea una **función** / una **clase** |
| `self` | Dentro de una clase, "este objeto" |
| `lambda x: ...` | Función corta en una línea |
| `if:` / `for` / `while` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

Los comentarios `#` explican cada línea; ejecuta de arriba a abajo.

## 1. El micro-motor que falla (reproducir el bug)

Este motor dispara **todas** las reglas que encajan, sin resolver conflictos: deduce **primero "aprobar" y luego "rechazar"** para el mismo pago.

In [1]:
class MicroMotor:
    def __init__(self):                 # al crear el objeto, prepara los datos
        self.hechos = {}                # hechos (dict)
        self.reglas = []                # reglas (list)
        self.traza  = []                # orden de disparo (list)

    def hecho(self, **h):               # añade hechos
        self.hechos.update(h)

    def regla(self, nombre, prioridad, condicion, accion):   # añade una regla
        self.reglas.append((nombre, prioridad, condicion, accion))

    def run(self):                      # ciclo: dispara TODO lo que encaja (sin elegir)
        pendientes = True
        while pendientes:
            pendientes = False
            for nombre, prioridad, condicion, accion in self.reglas:
                if nombre in self.traza:
                    continue
                if condicion(self.hechos):          # match
                    self.traza.append(nombre)        # (sin resolve)
                    accion(self.hechos)              # act
                    pendientes = True
        return self.hechos

motor = MicroMotor()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)
motor.regla("aprobar_pequena", 10, lambda h: h["importe"] < 500,
            lambda h: h.update(decision="aprobar"))
motor.regla("rechazar_reintentos", 10, lambda h: h["n_intentos"] >= 4,
            lambda h: h.update(decision="rechazar"))
print(motor.run())     # mira 'decision': acaba en 'rechazar', pero antes pasó por 'aprobar'
print(motor.traza)     # ambas reglas se dispararon -> la contradicción

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5, 'decision': 'rechazar'}
['aprobar_pequena', 'rechazar_reintentos']


El bug tiene **dos causas**: (1) no hay `resolve` (dispara todo lo que encaja) y (2) las reglas **no son excluyentes** (`aprobar` solo mira `importe`, no exige `n_intentos < 4`).

## 2. Solución: resolución de conflictos con `salience`

Añadimos la fase **resolver** (disparar solo la de mayor prioridad) y hacemos `aprobar` **excluyente** con `rechazar` (`importe < 500` **y** `n_intentos < 4`).

In [ ]:
class MicroMotor2:
    def __init__(self):
        self.hechos, self.reglas, self.traza = {}, [], []

    def hecho(self, **h):
        self.hechos.update(h)

    def regla(self, nombre, prioridad, condicion, accion):
        self.reglas.append((nombre, prioridad, condicion, accion))

    def run(self):
        while True:
            # 1) RECONOCER: reglas que encajan y aún no se han disparado
            activas = [(prioridad, nombre, accion) for nombre, prioridad, condicion, accion in self.reglas
                       if nombre not in self.traza and condicion(self.hechos)]
            if not activas:
                break
            # 2) RESOLVER: solo la de MAYOR prioridad (salience)
            _, nombre, accion = max(activas, key=lambda t: t[0])
            # 3) ACTUAR
            self.traza.append(nombre)
            accion(self.hechos)
        return self.hechos

motor = MicroMotor2()
motor.hecho(importe=200, antiguedad_meses=10, n_intentos=5)
# 'aprobar' ahora es excluyente: importe < 500 Y n_intentos < 4
motor.regla("aprobar_pequena", 20, lambda h: h["importe"] < 500 and h["n_intentos"] < 4,
            lambda h: h.update(decision="aprobar"))
motor.regla("rechazar_reintentos", 30,   # mayor salience: gana si hay conflicto
            lambda h: h["n_intentos"] >= 4,
            lambda h: h.update(decision="rechazar"))
print(motor.run())     # decision = rechazar (sin contradicción)
print(motor.traza)     # solo una regla

{'importe': 200, 'antiguedad_meses': 10, 'n_intentos': 5, 'decision': 'rechazar'}
['rechazar_reintentos']


### Cómo funciona `run()` (línea a línea)

```python
def run(self):
    while True:                      # bucle: repite hasta que no quede nada que disparar
        # 1) RECONOCER (match): reglas activas
        activas = [(p, n, a) for n, p, c, a in self.reglas
                   if n not in self.traza and c(self.hechos)]
        if not activas:              # si no queda ninguna activa, termina
            break
        # 2) RESOLVER (resolve): elegir la de MAYOR prioridad
        _, nombre, accion = max(activas, key=lambda t: t[0])
        # 3) ACTUAR (act)
        self.traza.append(nombre)    # anota que ya se disparó
        accion(self.hechos)          # ejecuta su acción (modifica los hechos)
    return self.hechos
```

- `activas = [...]` es una **comprensión de lista**: recorre `self.reglas` y se queda con las que **no están en `self.traza`** y cuya **`condicion(self.hechos)` es `True`**. De cada una guarda la tupla `(prioridad, nombre, accion)` — la **prioridad va primera**, para poder ordenar por ella.
- `max(activas, key=lambda t: t[0])` elige la tupla con **mayor prioridad** (el `lambda` compara por el primer elemento). Con `_` descartamos la prioridad y nos quedamos con `nombre` y `accion`.
- `self.traza.append(nombre)` registra la regla para que **no se dispare dos veces**.
- `accion(self.hechos)` ejecuta la acción (`h.update(decision=...)`), que **cambia los hechos** y puede activar otras reglas.

### ¿Por qué `aprobar` = 20 y `rechazar` = 30?

La **prioridad (salience)** decide qué regla gana cuando varias encajan a la vez: **gana el número más alto**. En un sistema de pagos, ante la duda conviene **rechazar** (es lo seguro: demasiados reintentos = posible fraude); por eso `rechazar` tiene 30 y `aprobar` 20. Los números concretos son arbitrarios: lo importante es que **30 > 20**. En `experta` usamos los mismos valores para que se comporten igual.

**Matiz:** en este caso las dos reglas ya son **mutuamente excluyentes** (`aprobar` exige `n_intentos < 4` y `rechazar` `>= 4`), así que con `n_intentos = 5` solo encaja `rechazar` y la salience ni se usa. Es la **red de seguridad** por si dos reglas llegaran a encajar a la vez.

Con `importe=200` y `n_intentos=5`, solo encaja `rechazar_reintentos` (porque `aprobar` exige `n_intentos < 4`). Si probaras `n_intentos=2`, encajaría `aprobar`. La `salience` (30 > 20) es la red de seguridad cuando dos reglas encajan a la vez.

## 3. La misma política con `experta`

In [3]:
import collections, collections.abc
if not hasattr(collections, "Mapping"):     # parche para Python 3.10+
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *

class Pagarium(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="evaluar")
    @Rule(Fact(accion="evaluar"), Fact(importe=P(lambda i: i < 500)),
          Fact(n_intentos=P(lambda n: n < 4)), salience=20)
    def aprobar(self):
        self.declare(Fact(decision="aprobar"))
    @Rule(Fact(n_intentos=P(lambda n: n >= 4)), salience=30)
    def rechazar(self):
        self.declare(Fact(decision="rechazar"))

motor = Pagarium(); motor.reset()
motor.declare(Fact(importe=200, n_intentos=5))
motor.run()
print([dict(f) for f in motor.facts.values() if "decision" in f])

[{'decision': 'rechazar', '__factid__': 3}]


## 4. La misma política en CLIPS (`clipspy`)

In [4]:
import clips
env = clips.Environment()
# UNA regla por cada env.build(...)
env.build("(defrule aprobar (importe ?i&:(< ?i 500)) (n_intentos ?n&:(< ?n 4)) => (assert (decision aprobar)))")
env.build("(defrule rechazar (declare (salience 30)) (n_intentos ?n&:(>= ?n 4)) => (assert (decision rechazar)))")
env.assert_string("(importe 200)")
env.assert_string("(n_intentos 5)")
env.run()
print([str(f) for f in env.facts()])

['(importe 200)', '(n_intentos 5)', '(decision rechazar)']


### Otro dominio: control de calefacción

In [5]:
motor = MicroMotor()
motor.hecho(temperatura=18, hora=23, ventana_abierta=False)
motor.regla("frio", 10, lambda h: h["temperatura"] < 20, lambda h: h.update(encender=True))
motor.regla("noche", 10, lambda h: h["hora"] >= 22, lambda h: h.update(modo="eco"))
motor.regla("ventilar", 5, lambda h: h["ventana_abierta"], lambda h: h.update(encender=False))
print(motor.run())
print(motor.traza)

{'temperatura': 18, 'hora': 23, 'ventana_abierta': False, 'encender': True, 'modo': 'eco'}
['frio', 'noche']


### Otro dominio: clasificación de animales

In [6]:
class Animales(KnowledgeEngine):
    @DefFacts()
    def inicio(self): yield Fact(analizar=True)
    @Rule(Fact(analizar=True), salience=10)
    def datos(self): self.declare(Fact(pelo=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))
    @Rule(Fact(pelo=True))
    def mamifero(self): self.declare(Fact(mamifero=True))
    @Rule(Fact(mamifero=True), Fact(carnivoro=True), Fact(color="leonado"), Fact(manchas="oscuras"))
    def guepardo(self): print("IDENTIFICADO: guepardo")

engine = Animales(); engine.reset(); engine.run()

IDENTIFICADO: guepardo


### Más reglas en CLIPS

In [7]:
env = clips.Environment()
env.build("(defrule pequena (importe ?i&:(< ?i 100)) => (assert (tarifa basica)))")
env.build("(defrule grande (importe ?i&:(>= ?i 100)) => (assert (tarifa premium)))")
env.assert_string("(importe 150)")
env.run()
print([str(f) for f in env.facts()])

['(importe 150)', '(tarifa premium)']


## Cierre: comparativa

| Motor | Resolución de conflictos | Salida con `n_intentos = 5` |
|---|---|---|
| `MicroMotor` (bug) | Ninguna | `aprobar` **y** `rechazar` |
| `MicroMotor2` (corregido) | `salience` | solo `rechazar` |
| `experta` | `salience` + reglas excluyentes | solo `rechazar` |
| `clipspy` (CLIPS) | `salience` + reglas excluyentes | solo `rechazar` |

**En 3 líneas:** el motor elige entre las reglas que encajan (resolver) con la `salience`; además, las reglas deben ser **mutuamente excluyentes** para no producir dos decisiones. Ese es el papel de la **resolución de conflictos**.